# Unsupervised exploration (_Iris_)

In [ ]:
from experiments.utils.constants import RANDOM_SEED
VERBOSE = True
DATASET_NAME = "Iris"
DATASET_ID = 53

## Dataset

In [ ]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

In [ ]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
del X

## Modeling

In [ ]:
from minisom_representation import calc_som_hyparams, SomRepresentation, plot_som_convergence_over_epochs

In [ ]:
# use helper methods to get SOM hyperparameter recommendations
print("Recommended SOM parameters:", calc_som_hyparams(X_scaled))
print("Recommended SOM parameters:", calc_som_hyparams(X_scaled, initial_sigma_factor=3.0))

In [ ]:
# define actual hyperparameters
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
d1, d2 = recommended_params.get("d1"), recommended_params.get("d2")
sigma = recommended_params.get("sigma")
print("d1 x d2:", d1, "x", d2)
print("sigma:", sigma)
decay_function = "linear_decay_to_zero"
fit_type = "online"
epoch = None

In [ ]:
# test candidate values for `num_iteration` hyperparameter
fig, errors_qe, errors_te = plot_som_convergence_over_epochs(
    SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=False, decay_function=decay_function),
    X_scaled,
    fit_type=fit_type, te_ceiling=.1,
    epoch_step_from=2, epoch_step_to=50, epoch_step=2,
    figsize=(10, 5), show_fig=True, verbose=VERBOSE
)
print(f"\nQE (first -> last): \t {errors_qe[0]:.2f} -> {errors_qe[-1]:.2f}")
print(f"TE (first -> last): \t {errors_te[0]:.2f} -> {errors_te[-1]:.2f}")

In [ ]:
# set selected `num_iteration` as epoch
epoch = 50

In [ ]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

## Inspection of the learned 2D topology

In [ ]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [ ]:
# traditional visuals
basin.legacy_pond().visualize_distance_map();
basin.legacy_pond().visualize_activation_map();

In [ ]:
# lilypond visuals
basin.pond() \
    .rhizome_layer() \
    .pad_layer(gap="nogap") \
    .petal_layer(min_size=8, max_size=25) \
    .visualize(width=1000, height=500);

## Register representation model in Databricks

In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
CATALOG = "workspace"
SCHEMA = "lilypond_experiments"
MODEL_NAME = "som-iris"
MODEL_PATH = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
MODEL_PATH

'workspace.lilypond_experiments.som-iris'

In [20]:
import mlflow
import pandas as pd
from typing import Any

EXPERIMENT_NAME = "/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_Iris"
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model:SomRepresentation, scaler):
        self.model = model
        self.scaler = scaler
    def predict(self, context, model_input, params: dict[str, Any] | None = None):
        """First transforms the input data via scaler, then predicts the winner node of the SOM."""
        return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

with mlflow.start_run():
    model = MLflowSomModelWrapper(som_rep, scaler)

    mlflow.log_metric("QE", som_rep.quantization_error)
    mlflow.log_metric("TE", som_rep.topographic_error)

    mlflow.pyfunc.log_model(
        python_model=model,
        name=MODEL_NAME,
        input_example=pd.DataFrame(X_scaled[:3]),
        pip_requirements=[
            "numpy",
            "pandas",
            "scikit-learn==1.5.2",
            "mlflow",
            "minisom",
        ],
        registered_model_name=MODEL_PATH
    )

2026/09/03 00:03:46 INFO mlflow.tracking.fluent: Experiment with name '/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_Iris' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
/opt/homebrew/Caskroom/miniconda/base/envs/wsom-lilypond-exp/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/09/03 00:03:57 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudP

🏃 View run languid-kite-211 at: dbc-ca44ccd5-3843.cloud.databricks.com/ml/experiments/1916796335158889/runs/86a3b79a779641539e976780cdee6545
🧪 View experiment at: dbc-ca44ccd5-3843.cloud.databricks.com/ml/experiments/1916796335158889


In [21]:
version = 1
registered_model = f"{MODEL_NAME}/{version}"
registered_model

'som-iris/1'

## Register metadata in Bianor

In [22]:
from databricks.connect import DatabricksSession
spark = DatabricksSession.builder.getOrCreate()

In [23]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [24]:
registered_model_location = f"{CATALOG}.{SCHEMA}.{registered_model}"
registered_model_location

'workspace.lilypond_experiments.som-iris/1'

In [25]:
bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t") # FIXME